# Grid Search Automatizado: A Busca pelo Melhor R²\n
Este notebook testa de forma autônoma várias combinações de `window`, `train_size` e `dropout`, salva todos os resultados numa tabela para análise e exporta automaticamente o melhor modelo e o melhor scaler.

In [ ]:
import os
import h5py
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib

## 1. Funções Base e Carregamento (Sem Deletar da RAM)\n
Como vamos fazer um loop, precisamos manter os dados brutos na memória.

In [ ]:
# Carregar arquivo
filename = 'data/N-CMAPSS_DS02-006.h5'
with h5py.File(filename, 'r') as hdf:
    W_dev = np.array(hdf.get('W_dev'))
    X_s_dev = np.array(hdf.get('X_s_dev'))
    Y_dev = np.array(hdf.get('Y_dev'))
    A_dev = np.array(hdf.get('A_dev'))
    
    W_test = np.array(hdf.get('W_test'))
    X_s_test = np.array(hdf.get('X_s_test'))
    Y_test = np.array(hdf.get('Y_test'))
    A_test = np.array(hdf.get('A_test'))

def create_temporal_features_safe(W, X_s, Y, A, window):
    matriz_base = np.concatenate((W, X_s), axis=1).astype('float32')
    df = pd.DataFrame(matriz_base)
    df['unit'] = A[:, 0].astype('float32') 
    df['RUL'] = Y.flatten().astype('float32')
    
    df_mean = df.groupby('unit').rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True).sort_index().astype('float32')
    df_var = df.groupby('unit').rolling(window=window, min_periods=1).var().fillna(0).reset_index(level=0, drop=True).sort_index().astype('float32')
    
    df_mean = df_mean[df.columns[:-2]]
    df_var = df_var[df.columns[:-2]]
    df_raw = pd.DataFrame(matriz_base)
    
    X_temporal = pd.concat([df_raw, df_mean, df_var], axis=1).values.astype('float32')
    y_labels = df['RUL'].values.astype('float32')
    
    del df, df_mean, df_var, df_raw, matriz_base
    gc.collect()
    
    return X_temporal, y_labels

## 2. O Loop Principal de Grid Search

In [ ]:
# Hiperparâmetros para testar
windows = [10, 20]
train_sizes = [0.30, 0.40, 0.50]
dropouts = [0.1, 0.2]

results = []
best_r2 = -float('inf')

# Callbacks que serão usados em todos
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=0)
stop_early = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

print("Iniciando Busca Automatizada...")

for w in windows:
    print(f"\n{'='*50}\n Gerando features para JANELA = {w}...")
    X_train_raw, y_train_full = create_temporal_features_safe(W_dev, X_s_dev, Y_dev, A_dev, window=w)
    X_test_raw, y_test = create_temporal_features_safe(W_test, X_s_test, Y_test, A_test, window=w)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw).astype('float32')
    X_test_scaled = scaler.transform(X_test_raw).astype('float32')
    
    del X_train_raw, X_test_raw
    gc.collect()
    
    for t_size in train_sizes:
        print(f"  -> Amostrando {t_size*100}% dos dados...")
        X_train_shuf, _, y_train_shuf, _ = train_test_split(
            X_train_scaled, y_train_full, train_size=t_size, random_state=42
        )
        
        for drop in dropouts:
            print(f"    --> Treinando MLP com Dropout = {drop}...")
            
            # Arquitetura
            model = Sequential()
            model.add(tf.keras.layers.Input(shape=(X_train_shuf.shape[1],)))
            model.add(Dense(units=256, activation='relu'))
            model.add(BatchNormalization())
            model.add(Dropout(drop))
            model.add(Dense(units=128, activation='relu'))
            model.add(BatchNormalization())
            model.add(Dropout(drop))
            model.add(Dense(units=64, activation='relu'))
            model.add(BatchNormalization())
            model.add(Dense(1, activation='linear'))
            
            model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
            
            # Treinamento Silencioso (verbose=0 para não sujar o terminal do Colab)
            history = model.fit(
                X_train_shuf, y_train_shuf,
                validation_split=0.2, 
                epochs=60, # Limite menor por segurança no Colab
                batch_size=4096, 
                callbacks=[stop_early, reduce_lr],
                verbose=0
            )
            
            # Avaliação
            y_pred = model.predict(X_test_scaled, verbose=0)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            
            print(f"        [RESULTADO] MAE: {mae:.2f} | R²: {r2:.4f}")
            
            # Armazenando no relatório
            results.append({
                'Window': w,
                'Train_Size': t_size,
                'Dropout': drop,
                'MAE': mae,
                'R2': r2
            })
            
            # Salvando se for o melhor absoluto
            if r2 > best_r2:
                best_r2 = r2
                model.save('data/modelo_VENCEDOR_gridsearch.keras')
                joblib.dump(scaler, 'data/scaler_VENCEDOR_gridsearch.pkl')
                print("        ⭐ NOVO RECORDE ALCANÇADO! Modelo Salvo.")
                
        # Fim do loop de Dropout
        del X_train_shuf, y_train_shuf
        gc.collect()
        
    # Fim do loop de Train Size
    del X_train_scaled, X_test_scaled, y_train_full, y_test
    gc.collect()

## 3. Relatório Final de Desempenho

In [ ]:
print("\n
" + "="*50)
print("🏆 RESUMO DE TODOS OS TESTES 🏆")
print("="*50)

df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='R2', ascending=False).reset_index(drop=True)
display(df_results)

best_config = df_results.iloc[0]
print(f"\n
O Melhor modelo obteve R² de {best_config['R2']:.4f} com a seguinte configuração:")
print(f"- Janela (Window): {best_config['Window']}")
print(f"- Amostragem (Train_Size): {best_config['Train_Size']*100}%")
print(f"- Dropout: {best_config['Dropout']}")
print("\n
O arquivo 'modelo_VENCEDOR_gridsearch.keras' já está salvo na pasta 'data/'!")